In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from category_encoders import BinaryEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder
from category_encoders import TargetEncoder, BinaryEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.feature_selection import mutual_info_classif, chi2, f_classif, SelectKBest
from sklearn.metrics import roc_auc_score, classification_report
import pickle

## Data familiarization

In [2]:
df_orgin = pd.read_csv(r"Data V2\aqarmap_final.csv")
df = df_orgin.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4753 entries, 0 to 4752
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Listing_ID       4744 non-null   str    
 1   Type             4753 non-null   str    
 2   Address          4753 non-null   str    
 3   Price            4753 non-null   float64
 4   Price_Per_M2     4739 non-null   float64
 5   Area             4685 non-null   float64
 6   Rooms            4314 non-null   str    
 7   Bathrooms        4273 non-null   str    
 8   Floor            3853 non-null   float64
 9   Year_Built       4023 non-null   float64
 10  Finish           4753 non-null   int64  
 11  View             4702 non-null   str    
 12  Seller_Type      4282 non-null   str    
 13  Amenities        3230 non-null   str    
 14  Description      1248 non-null   str    
 15  Link             4753 non-null   str    
 16  Amenities_Count  4753 non-null   int64  
 17  Is_Installments  4753 non

In [3]:
df.head()

,Listing_ID,Type,Address,Price,Price_Per_M2,Area,Rooms,Bathrooms,Floor,Year_Built,Finish,View,Seller_Type,Amenities,Description,Link,Amenities_Count,Is_Installments
0,EG-6901132,Villa,north coast resorts riviera,6800000.0,19429.0,350.0,4 rooms,3 bathroom,NaN,2009.0,1,Garden,Developer,Garden,Villa For sale in Riviera,https://aqarmap.com.eg/en/listing/6901132-for-...,1,1
1,EG-6671452,Apartment,cairo 6th of october compounds murooj,5000000.0,37594.0,133.0,3 rooms,2 bathroom,2.0,2026.0,1,Main Street,Developer,NaN,Apartment For sale in Murooj Compound - Dar El...,https://aqarmap.com.eg/en/listing/6671452-for-...,0,0
2,EG-6903778,Land,cairo 6th of october ltws t lshrqy,5500000.0,5314.0,1035.0,NaN,NaN,NaN,NaN,1,Main Street,Broker,NaN,Land For sale in Eastern Expansions with size ...,https://aqarmap.com.eg/en/listing/6903778-for-...,0,0
3,EG-6897924,Chalet,north coast resorts solare resort misr italia,9974100.0,75561.0,132.0,2 rooms,2 bathroom,0.0,2029.0,1,Lake,Private Owner,"Security, Garden, Solar",Directly from the ownerA ground-floor chalet w...,https://aqarmap.com.eg/en/listing/6897924-for-...,3,1
4,EG-6898957,Apartment,cairo new heliopolis lhy lthny,1600000.0,13333.0,120.0,2 rooms,3 bathroom,3.0,2025.0,1,Main Street,Developer,NaN,Apartments For sale with size 120 M² Semi Fini...,https://aqarmap.com.eg/en/listing/6898957-for-...,0,1


In [4]:
df.describe()

,Price,Price_Per_M2,Area,Floor,Year_Built,Finish,Amenities_Count,Is_Installments
count,4.753000e+03,4739.000000,4685.000000,3853.000000,4023.000000,4753.000000,4753.000000,4753.000000
mean,1.051528e+07,53240.047056,267.528922,2.879315,2021.487199,0.989691,2.128340,0.464969
std,1.956309e+07,49829.591510,1678.995921,2.930968,9.328540,0.101021,1.996193,0.498824
min,3.500000e+03,20.000000,0.000000,0.000000,1925.000000,0.000000,0.000000,0.000000
25%,3.612000e+06,24615.000000,120.000000,1.000000,2020.000000,1.000000,0.000000,0.000000
50%,6.000000e+06,37267.000000,160.000000,2.000000,2025.000000,1.000000,1.000000,0.000000
75%,1.045000e+07,62989.500000,220.000000,4.000000,2026.000000,1.000000,4.000000,1.000000
max,5.999490e+08,617480.000000,63000.000000,45.000000,2037.000000,1.000000,7.000000,1.000000


In [5]:
df.isnull().sum()

Listing_ID            9
Type                  0
Address               0
Price                 0
Price_Per_M2         14
Area                 68
Rooms               439
Bathrooms           480
Floor               900
Year_Built          730
Finish                0
View                 51
Seller_Type         471
Amenities          1523
Description        3505
Link                  0
Amenities_Count       0
Is_Installments       0
dtype: int64

In [6]:
print(df.shape[0])
df = df.drop_duplicates(subset=["Listing_ID"])
print(df.shape[0])

4753
4725


In [7]:
for col in df.select_dtypes(include=["object"]).columns:
    print(f"{col} unique_count: {df[col].nunique()}, \n top_values: {df[col].value_counts().head(10).to_dict()}")

Listing_ID unique_count: 4724, 
 top_values: {'EG-6901132': 1, 'EG-6671452': 1, 'EG-6903778': 1, 'EG-6897924': 1, 'EG-6898957': 1, 'EG-6859550': 1, 'EG-6899863': 1, 'EG-6723602': 1, 'EG-6816744': 1, 'EG-6547418': 1}
Type unique_count: 13, 
 top_values: {'Apartment': 2977, 'Duplex': 536, 'Villa': 416, 'Chalet': 181, 'Roof': 154, 'Penthouse': 146, 'Office': 76, 'Studio': 67, 'Commercial': 60, 'Land': 58}
Address unique_count: 1563, 
 top_values: {'red sea hurghada city el hadba sheraton st': 59, 'red sea hy lkwthr': 42, 'alexandria smouha Muruj': 39, 'cairo 6th of october el tawsaat el shamalya hy 2000 qt': 36, 'the med': 36, 'kite residence': 34, 'isola compound': 34, 'north coast resorts virginia plaza resort solid': 33, 'cairo 6th of october hadaeq october compounds sun capital': 31, 'cairo new cairo bait el watan fourth neighborhood': 30}
Rooms unique_count: 78, 
 top_values: {'3 rooms': 2131, '2 rooms': 867, '4 rooms': 408, '1 rooms': 338, '5 rooms': 130, '3': 62, '6 rooms': 54, '2'

C:\Users\amrta\AppData\Local\Temp\ipykernel_18076\1971780853.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:


## Amenities column

The `Amenities` column is a raw comma-separated string (ex: `"Pool, Elevator, Security"`). 
Instead of one-hot encoding each amenity (sparse + noisy), we group them into 4 semantic binary flags

In [8]:
df["Amenities"].value_counts()

Amenities
Garden                                                     737
Elevator, Water Meter, Security, Swimming Pool, Balcony    241
Elevator, Water Meter, Security, Balcony                   165
Security, Swimming Pool, Garden                            158
Elevator, Water Meter, Security, Garden, Balcony           141
                                                          ... 
Water Meter, Natural Gas, Security, Parking, Garden          1
Elevator, Natural Gas, Parking, Garden, Balcony              1
Elevator, Swimming Pool, Parking                             1
Natural Gas, Garden                                          1
Elevator, Water Meter, Swimming Pool, Garden, Balcony        1
Name: count, Length: 190, dtype: int64

In [9]:
amenity_groups = {
    "has_utilities": ["Electricity Meter", "Water Meter", "Natural Gas"],
    "has_security": ["Security"],
    "has_luxury": ["Pool", "Garden", "Solar"],
    "has_access": ["Elevator", "Parking"],
}


def check_group(amenity_string, keywords):
    if pd.isna(amenity_string):
        return 0
    return int(any(kw in amenity_string for kw in keywords))


for group, keywords in amenity_groups.items():
    df[group] = df["Amenities"].apply(lambda x: check_group(x, keywords))

df.info()

<class 'pandas.DataFrame'>
Index: 4725 entries, 0 to 4752
Data columns (total 22 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Listing_ID       4724 non-null   str    
 1   Type             4725 non-null   str    
 2   Address          4725 non-null   str    
 3   Price            4725 non-null   float64
 4   Price_Per_M2     4718 non-null   float64
 5   Area             4657 non-null   float64
 6   Rooms            4293 non-null   str    
 7   Bathrooms        4252 non-null   str    
 8   Floor            3844 non-null   float64
 9   Year_Built       4023 non-null   float64
 10  Finish           4725 non-null   int64  
 11  View             4682 non-null   str    
 12  Seller_Type      4282 non-null   str    
 13  Amenities        3208 non-null   str    
 14  Description      1248 non-null   str    
 15  Link             4725 non-null   str    
 16  Amenities_Count  4725 non-null   int64  
 17  Is_Installments  4725 non-null

In [10]:
df = df.drop(["Amenities"], axis=1)

## Address column

The `Address` field is free-text and inconsistent. We first normalize the string, then apply two lookup strategies in order: (1) regex pattern matching against known city names and aliases, (2) a manual project→city map for branded compounds whose names contain no geographic hint.

In [11]:
df["Address"].value_counts()

Address
red sea hurghada city el hadba sheraton st                           59
red sea hy lkwthr                                                    42
alexandria smouha Muruj                                              39
cairo 6th of october el tawsaat el shamalya hy 2000 qt               36
the med                                                              36
                                                                     ..
cairo nasr city 9th zone hossam el deen basiouny st                   1
cairo new administrative capital r8 moraya edge stone                 1
cairo new cairo south investors el nasr st                            1
cairo 6th of october hadaeq october kmbwnd fy hdyq ktwbr dar misr     1
cairo el sheikh zayed city compounds beverly hills                    1
Name: count, Length: 1563, dtype: int64

In [12]:
noise_words = [
    "city",
    "compound",
    "compounds",
    "hy",
    "mdyn",
    "kmbwnd",
    "lstd",
    "jdyl",
    "el",
    "al",
    "of",
    "the",
    "kwrnysh",
    "lnyl",
]


def clean_address(x):
    x = str(x).lower()
    x = re.sub(r"[^a-z\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    words = x.split()
    words = [w for w in words if w not in noise_words]
    return " ".join(words[:4])


df["Address"] = df["Address"].apply(clean_address)

df["Address"].value_counts()

Address
cairo new administrative capital    265
cairo th october hadaeq             225
red sea hurghada resorts            137
cairo new cairo bait                120
cairo th october ahyaa              120
                                   ... 
north coast syd bd                    1
cairo th october aeon                 1
alexandria smouha mahmoudya road      1
alexandria king maryot jezzine        1
cairo sheikh zayed beverly            1
Name: count, Length: 800, dtype: int64

In [13]:
CITY_PATTERNS = {
    "new capital": [
        "new capital",
        "administrative",
        "capital",
        "eins park",
        "inizio",
        "track rev",
        "r[0-9]",
    ],
    "new cairo": [
        "new cairo",
        "tagamoa",
        "tagammo",
        "tagamo3",
        "fifth settlement",
        "5th settlement",
        "belagio",
        "at nine",
        "beta greens",
    ],
    "october": [
        "6th of october",
        "6 october",
        "october",
        "th october",
        "ashgaar",
        "isola",
        "kite residence",
        "calm residence",
        "v levels",
        "belong",
    ],
    "sheikh zayed": ["sheikh zayed", "zayed"],
    "madinaty": ["madinaty"],
    "maadi": ["maadi"],
    "rehab": ["rehab"],
    "shorouk": ["shorouk"],
    "heliopolis": ["heliopolis", "masr el gedida"],
    "nasr city": ["nasr city", "nasr"],
    "cairo": [
        "dokki",
        "mohandesen",
        "faisal",
        "imbaba",
        "helwan",
        "mansuriyyah",
        "talbeya",
        "maryotyah",
        "tahrir",
    ],
    "alexandria": [
        "alexandria",
        "smouha",
        "fleming",
        "lauran",
        "moharram bey",
        "sidi gaber",
    ],
    "north coast": ["north coast", "sahel"],
    "hurghada": ["hurghada", "red sea"],
    "ain sokhna": [
        "ain sokhna",
        "elsokhna",
        "sokhna",
        "porto sokhna",
        "azha",
        "stella",
        "la vista",
        "mountain view sokhna",
        "il monte galala",
        "galala",
    ],
    "sharm sheikh": ["sharm sheikh"],
    "ras sidr": ["ras sidr"],
    "marsa matruh": ["marsa matruh"],
    "damietta": ["damietta"],
    "gharbia": ["gharbia"],
    "dakahlia": ["dakahlia"],
    "port said": ["port said"],
    "monufia": ["monufia"],
    "sharqia": ["sharqia"],
    "fayoum": ["fayoum"],
    "ismailia": ["ismailia"],
    "qalyubia": ["qalyubia"],
    "suez": ["suez"],
    "beheira": ["beheira", "damanhour"],
    "kafr sheikh": ["kafr sheikh"],
    "aswan": ["aswan"],
    "sohag": ["sohag"],
    "minia": ["minia"],
    "asyut": ["asyut"],
    "qina": ["qina"],
    "beni suef": ["beni suef"],
    "sinai": ["sinai", "sina"],
}

In [14]:
PROJECT_TO_CITY = {
    # October / Zayed
    "pyramids heights": "october",
    "park point": "october",
    "naia west": "sheikh zayed",
    "advida": "october",
    "greens": "october",
    #
    "golden park": "cairo",
    "guzal": "monufia",
    # New Capital
    "serrano": "new capital",
    "lark residence": "new capital",
    "novalist": "new capital",
    "cloud business complex": "new capital",
    "hills one": "new capital",
    "za mall": "new capital",
    "acacia mall": "new capital",
    "juno plaza": "new capital",
    "kardia": "new capital",
    "zalink": "new capital",
    "catalan": "new capital",
    # North Coast
    "lasirena": "north coast",
    "rivira beach": "north coast",
    "verona": "north coast",
    "celebration west beach": "north coast",
    "med": "north coast",
    # Alexandria
    "king mariout": "alexandria",
    "makani king mariout": "alexandria",
    # Sokhna
    "groove": "ain sokhna",
    "baymount": "ain sokhna",
    "panorama hills": "ain sokhna",
    "steigenberger": "ain sokhna",
    "porto": "ain sokhna",
    "stella": "ain sokhna",
    "azha": "ain sokhna",
    "blue": "ain sokhna",
    "heaven": "ain sokhna",
    "galala": "ain sokhna",
    # noise
    "https": "noise",
    "aqarmap": "noise",
}

In [15]:
def match_pattern(text, pattern):
    return re.search(rf"\b{pattern}\b", text)

In [16]:
def extract_city(x):
    x = str(x).lower()
    x = re.sub(r"[^a-z\s]", " ", x)

    for city, patterns in CITY_PATTERNS.items():
        for p in patterns:
            if match_pattern(x, p):
                return city

    if "alexandria" in x:
        return "alexandria"

    if "cairo" in x:
        return "cairo"

    if any(n in x for n in ["https", "aqarmap"]):
        return "unknown"

    for proj, city in PROJECT_TO_CITY.items():
        if proj in x:
            if city == "noise":
                return "unknown"
            return city

    return "unknown"

In [17]:
df["City"] = df["Address"].apply(extract_city)
df["City"].value_counts()

City
new cairo       1338
october          868
cairo            602
new capital      336
hurghada         275
north coast      226
sheikh zayed     223
alexandria       179
nasr city        143
unknown           92
heliopolis        86
ain sokhna        81
maadi             61
shorouk           50
damietta          27
gharbia           24
dakahlia          15
marsa matruh      12
sharqia           11
qalyubia          11
ismailia           9
port said          8
sohag              8
monufia            7
suez               5
ras sidr           5
sharm sheikh       4
aswan              4
qina               4
fayoum             3
asyut              2
beheira            2
sinai              1
minia              1
beni suef          1
kafr sheikh        1
Name: count, dtype: int64

In [18]:
df = df.drop(["Address"], axis=1)

## Rooms

`Rooms` and `Bathrooms` are stored as strings like `"3 Rooms"`. We extract the integer, cap at 20 to discard garbage values, then drop the original string columns.

In [19]:
def extract_num(s):
    if pd.isna(s):
        return np.nan

    s = str(s).lower()

    match = re.search(r"(\d+)\s*(room|bed|bath)", s)
    if match:
        val = int(match.group(1))
        return val if val <= 20 else np.nan

    match = re.search(r"\b(\d+)\b", s)
    if match:
        val = int(match.group(1))
        return val if val <= 20 else np.nan

    return np.nan


df["Rooms_Count"] = df["Rooms"].apply(extract_num)
df["Bathrooms_Count"] = df["Bathrooms"].apply(extract_num)
df.info()

<class 'pandas.DataFrame'>
Index: 4725 entries, 0 to 4752
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Listing_ID       4724 non-null   str    
 1   Type             4725 non-null   str    
 2   Price            4725 non-null   float64
 3   Price_Per_M2     4718 non-null   float64
 4   Area             4657 non-null   float64
 5   Rooms            4293 non-null   str    
 6   Bathrooms        4252 non-null   str    
 7   Floor            3844 non-null   float64
 8   Year_Built       4023 non-null   float64
 9   Finish           4725 non-null   int64  
 10  View             4682 non-null   str    
 11  Seller_Type      4282 non-null   str    
 12  Description      1248 non-null   str    
 13  Link             4725 non-null   str    
 14  Amenities_Count  4725 non-null   int64  
 15  Is_Installments  4725 non-null   int64  
 16  has_utilities    4725 non-null   int64  
 17  has_security     4725 non-null

In [20]:
df = df.drop(["Rooms", "Bathrooms"], axis=1)

## Feature Engineering

In [21]:
# Temporal Logic (Off-plan vs Ready)
current_year = 2025
df["Is_Off_Plan"] = (df["Year_Built"] > current_year).astype(int)
df["Years_to_Delivery"] = (df["Year_Built"] - current_year).clip(lower=0)
df["Property_Age"] = (current_year - df["Year_Built"]).clip(lower=0)

**Temporal features:** `Year_Built` alone is a weak signal. Breaking it into `Is_Off_Plan`, `Years_to_Delivery`, and `Property_Age` lets the model treat future deliveries and aged stock differently.

In [22]:
# Luxury
df["Total_Rooms"] = df["Rooms_Count"] + df["Bathrooms_Count"]
df["Avg_Room_Size"] = df["Area"] / df["Total_Rooms"].replace(0, np.nan)
df["Area_Per_Room"] = df["Area"] / df["Rooms_Count"].replace(0, np.nan)
df["Rooms_per_Bathroom"] = df["Rooms_Count"] / (df["Bathrooms_Count"] + 1).replace(
    0, np.nan
)

**Size-derived features:** Raw area is less informative than area relative to room count. These ratios act as a proxy for spaciousness and help distinguish luxury units from cramped ones at the same total area.

## Outliers

IQR removal is applied **per property type** rather than globally. Since studio and a villa have very different price distributions (a global IQR would flag normal villas as outliers).

In [23]:
def remove_type_outliers(group):
    q1 = group["Price"].quantile(0.25)
    q3 = group["Price"].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return group[(group["Price"] >= lower) & (group["Price"] <= upper)]


df_final = df.copy()

mask = pd.concat([remove_type_outliers(group) for _, group in df_final.groupby("Type")])
df_final = df_final.loc[mask.index].reset_index(drop=True)

In [24]:
cols_to_drop = [
    "Listing_ID",
    "Description",
    "Link",
    "Price_Per_M2",
]
df_final = df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns])

In [25]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 4459 entries, 0 to 4458
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Type                4459 non-null   str    
 1   Price               4459 non-null   float64
 2   Area                4415 non-null   float64
 3   Floor               3652 non-null   float64
 4   Year_Built          3824 non-null   float64
 5   Finish              4459 non-null   int64  
 6   View                4419 non-null   str    
 7   Seller_Type         4071 non-null   str    
 8   Amenities_Count     4459 non-null   int64  
 9   Is_Installments     4459 non-null   int64  
 10  has_utilities       4459 non-null   int64  
 11  has_security        4459 non-null   int64  
 12  has_luxury          4459 non-null   int64  
 13  has_access          4459 non-null   int64  
 14  City                4459 non-null   str    
 15  Rooms_Count         4066 non-null   float64
 16  Bathrooms_Count  

In [26]:
df_final.isna().sum()

Type                    0
Price                   0
Area                   44
Floor                 807
Year_Built            635
Finish                  0
View                   40
Seller_Type           388
Amenities_Count         0
Is_Installments         0
has_utilities           0
has_security            0
has_luxury              0
has_access              0
City                    0
Rooms_Count           393
Bathrooms_Count       473
Is_Off_Plan             0
Years_to_Delivery     635
Property_Age          635
Total_Rooms           507
Avg_Room_Size         518
Area_Per_Room         411
Rooms_per_Bathroom    507
dtype: int64

## Data Splitting 

Split **before** encoding and imputing. Fitting any transformer on the full dataset leaks test distribution into training. Everything downstream is fit on `X_train` only.

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    df_final.drop("Price", axis=1), df_final["Price"], test_size=0.25, random_state=42
)

In [28]:
numeric_features = X_train.select_dtypes("number").columns
categorical_features = X_train.select_dtypes(exclude="number").columns

## Encode, Impute & Scale

**Encoding strategy:**
- `City` → `TargetEncoder` with `smoothing=10`: ~30 cities makes this high cardinality. Mean-target encoding is informative, and smoothing shrinks rare cities toward the global mean to avoid overfitting.
- `Type`, `View`, `Seller_Type` → `BinaryEncoder`: compact numeric representation without imposing ordinal assumptions.

In [29]:
target_cols = ["City"]
binary_cols = ["Type", "View", "Seller_Type"]

target_enc = TargetEncoder(cols=target_cols, smoothing=10)
X_train[target_cols] = target_enc.fit_transform(X_train[target_cols], y_train)
X_test[target_cols] = target_enc.transform(X_test[target_cols])

for col in binary_cols:
    missing_train = X_train[col].isna()
    missing_test = X_test[col].isna()

    enc = BinaryEncoder(cols=[col], drop_invariant=True)
    X_train_enc = enc.fit_transform(X_train[[col]])
    X_test_enc = enc.transform(X_test[[col]])

    X_train_enc.loc[missing_train] = np.nan
    X_test_enc.loc[missing_test] = np.nan

    X_train = pd.concat([X_train.drop(col, axis=1), X_train_enc], axis=1)
    X_test = pd.concat([X_test.drop(col, axis=1), X_test_enc], axis=1)

X_train.head()

,Area,Floor,Year_Built,Finish,Amenities_Count,Is_Installments,has_utilities,has_security,has_luxury,has_access,...,Type_1,Type_2,Type_3,View_0,View_1,View_2,View_3,Seller_Type_0,Seller_Type_1,Seller_Type_2
3647,51.0,2.0,2028.0,1,0,1,0,0,0,0,...,0,0,1,0.0,0.0,0.0,1.0,0.0,0.0,1.0
865,251.0,1.0,2026.0,1,0,1,0,0,0,0,...,0,1,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
908,140.0,4.0,2026.0,1,0,1,0,0,0,0,...,0,1,0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4322,400.0,NaN,2020.0,1,0,0,0,0,0,0,...,0,1,1,0.0,0.0,1.0,1.0,0.0,0.0,1.0
2093,250.0,1.0,2022.0,1,0,0,0,0,0,0,...,0,1,0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


`IterativeImputer` with a RandomForest estimator imputes each missing feature as a function of the others (more accurate than mean/median).

In [30]:
impute = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=100,
        max_features=0.5,
        random_state=42,
        n_jobs=-1,
    ),
    max_iter=20,
    tol=0.01,
    initial_strategy="median",
    random_state=42,
)

X_train_processed = impute.fit_transform(X_train)
X_test_processed = impute.transform(X_test)
X_train_processed

array([[5.100e+01, 2.000e+00, 2.028e+03, ..., 0.000e+00, 0.000e+00,
        1.000e+00],
       [2.510e+02, 1.000e+00, 2.026e+03, ..., 0.000e+00, 1.000e+00,
        0.000e+00],
       [1.400e+02, 4.000e+00, 2.026e+03, ..., 0.000e+00, 1.000e+00,
        0.000e+00],
       ...,
       [3.900e+02, 1.000e+00, 2.026e+03, ..., 0.000e+00, 1.000e+00,
        0.000e+00],
       [1.330e+02, 3.000e+00, 2.021e+03, ..., 0.000e+00, 0.000e+00,
        1.000e+00],
       [1.100e+02, 2.000e+00, 2.027e+03, ..., 0.000e+00, 1.000e+00,
        0.000e+00]])

In [31]:
binary_flag_cols = [
    "has_utilities",
    "has_security",
    "has_luxury",
    "has_access",
    "Is_Installments",
    "Finish",
]

binary_encoded_cols = [
    col
    for col in X_train.columns
    if any(col.startswith(base) for base in ["Type_", "View_", "Seller_"])
]

cols_to_exclude = binary_flag_cols + binary_encoded_cols

cols_to_scale = [
    col
    for col in X_train.columns
    if col not in cols_to_exclude and X_train[col].dtype in ["float64", "int64"]
]
cols_to_scale

['Area',
 'Floor',
 'Year_Built',
 'Amenities_Count',
 'City',
 'Rooms_Count',
 'Bathrooms_Count',
 'Years_to_Delivery',
 'Property_Age',
 'Total_Rooms',
 'Avg_Room_Size',
 'Area_Per_Room',
 'Rooms_per_Bathroom']

In [32]:
scaler = StandardScaler()
X_train_processed_df = pd.DataFrame(
    X_train_processed, columns=X_train.columns, index=X_train.index
)
X_test_processed_df = pd.DataFrame(
    X_test_processed, columns=X_train.columns, index=X_test.index
)

X_train_processed_df[cols_to_scale] = scaler.fit_transform(
    X_train_processed_df[cols_to_scale]
)
X_test_processed_df[cols_to_scale] = scaler.transform(
    X_test_processed_df[cols_to_scale]
)

X_train_processed_df.head()

,Area,Floor,Year_Built,Finish,Amenities_Count,Is_Installments,has_utilities,has_security,has_luxury,has_access,...,Type_1,Type_2,Type_3,View_0,View_1,View_2,View_3,Seller_Type_0,Seller_Type_1,Seller_Type_2
3647,-0.265935,-0.324330,0.685529,1.0,-1.061759,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
865,0.091513,-0.714972,0.464158,1.0,-1.061759,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
908,-0.106871,0.456953,0.464158,1.0,-1.061759,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4322,0.357811,-0.027443,-0.199952,1.0,-1.061759,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
2093,0.089726,-0.714972,0.021418,1.0,-1.061759,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [33]:
# Pickle
with open("X_train_processed.pkl", "wb") as f:
    pickle.dump(X_train_processed_df, f)

with open("X_test_processed.pkl", "wb") as f:
    pickle.dump(X_test_processed_df, f)

with open("y_train.pkl", "wb") as f:
    pickle.dump(y_train, f)

with open("y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)

# CSV
X_train_processed_df.to_csv("X_train_processed.csv", index=False)
X_test_processed_df.to_csv("X_test_processed.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Exported")

Exported
